# Course 11: Mathematics for Machine Learning
## Session 6: Probability for ML

**Goal:** compute expectations, variances and conditional probabilities by hand and by simulation, watch averages converge, work through Bayes' rule with a medical test, sample a 2-D Gaussian with its Cholesky factor and draw its covariance ellipses, and see a confounder create correlation.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. Random variables, expectation and variance
A **random variable** is a number decided by chance: the face of a die, a label that is 1 with probability $p$ (a **Bernoulli** variable), a class drawn from probabilities $(p_1, \dots, p_K)$ (a **categorical** variable, what a softmax outputs), or a measurement with **Gaussian** noise. For a discrete variable,
$$E[X] = \sum_x x\, P(X = x), \qquad \operatorname{Var}[X] = E[(X - E[X])^2] = E[X^2] - E[X]^2 .$$

**By hand first:** for a fair die, compute $E[X]$, $E[X^2]$ and $\operatorname{Var}[X]$ as fractions. For a Bernoulli variable with $p = 0.3$, what are the mean and the variance $p(1 - p)$?

In [ ]:
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

faces = range(1, 7)
E = sum(Fraction(k, 6) for k in faces)
E2 = sum(Fraction(k * k, 6) for k in faces)
print(f"fair die: E[X] = {E}, E[X^2] = {E2}, Var[X] = {E2 - E ** 2} = {float(E2 - E ** 2):.4f}, sd = {float(E2 - E ** 2) ** 0.5:.4f}")
p = 0.3
print(f"Bernoulli(0.3): mean {p}, variance p(1 - p) = {p * (1 - p):.2f}")

rolls = rng.integers(1, 7, size=100_000)
coins = (rng.random(100_000) < p).astype(int)
print(f"simulated, 100,000 draws: die mean {rolls.mean():.4f}, variance {rolls.var():.4f}; Bernoulli mean {coins.mean():.4f}, variance {coins.var():.4f}")

probs = np.array([0.5, 0.3, 0.2])                     # a categorical variable, like a softmax output
u = rng.random(10_000)
draws = np.searchsorted(np.cumsum(probs), u, side="right")   # inverse-CDF sampling: where u falls in (0.5, 0.8, 1.0)
print("categorical (0.5, 0.3, 0.2), frequencies in 10,000 draws:", np.bincount(draws, minlength=3) / 10_000)
g = rng.normal(loc=2.0, scale=3.0, size=100_000)
print(f"Gaussian with mean 2 and standard deviation 3: sample mean {g.mean():.4f}, sample variance {g.var():.4f} (true 9)")

### 2. The law of large numbers, by simulation
The average of $n$ independent draws approaches $E[X]$ as $n$ grows (the **law of large numbers**). Its spread shrinks like $\sigma/\sqrt n$, because $\operatorname{Var}[\bar X_n] = \sigma^2/n$ for independent draws with variance $\sigma^2$. In machine learning, a minibatch gradient and a test accuracy are both averages of this kind.

**By hand first:** the die has $\sigma = \sqrt{35/12} \approx 1.708$. What is $\sigma/\sqrt n$ for $n$ = 100 and 10,000?

In [ ]:
sigma = (35 / 12) ** 0.5
reps = rng.integers(1, 7, size=(2000, 10_000))      # 2,000 independent experiments of 10,000 rolls each
print("    n   sd of the mean over 2,000 experiments   sigma / sqrt(n)   share within 2 sigma / sqrt(n) of 3.5")
for n in [10, 100, 1000, 10_000]:
    means = reps[:, :n].mean(axis=1)
    print(f"{n:6d}   {means.std():37.4f}   {sigma / np.sqrt(n):15.4f}   {np.mean(np.abs(means - 3.5) < 2 * sigma / np.sqrt(n)):10.3f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.3))
ns = np.arange(1, 10_001)
for r in range(5):
    ax[0].plot(ns, np.cumsum(reps[r]) / ns, lw=0.8)
ax[0].plot(ns, 3.5 + 2 * sigma / np.sqrt(ns), "k--", lw=0.8, label="3.5 +/- 2 sigma / sqrt(n)")
ax[0].plot(ns, 3.5 - 2 * sigma / np.sqrt(ns), "k--", lw=0.8)
ax[0].set_xscale("log"); ax[0].set_ylim(1.5, 5.5); ax[0].legend(fontsize=8)
ax[0].set(title="Running mean of die rolls (5 runs)", xlabel="number of rolls n", ylabel="average so far")
n_grid = np.array([10, 30, 100, 300, 1000, 3000, 10_000])
ax[1].loglog(n_grid, [reps[:, :n].mean(axis=1).std() for n in n_grid], "o", label="simulated")
ax[1].loglog(n_grid, sigma / np.sqrt(n_grid), "k-", lw=0.8, label="sigma / sqrt(n)")
ax[1].legend(fontsize=8)
ax[1].set(title="Spread of the mean shrinks like 1/sqrt(n)", xlabel="number of rolls n", ylabel="standard deviation of the mean")
plt.tight_layout(); plt.show()

### 3. Joint, marginal and conditional probability
A **joint** distribution $P(A, B)$ gives the probability of each pair of outcomes. Summing over one variable gives the **marginal** of the other, $P(A) = \sum_b P(A, b)$, and the **conditional** probability is $P(A \mid B) = P(A, B)/P(B)$. $A$ and $B$ are **independent** when $P(A, B) = P(A)P(B)$ for every pair of outcomes.

Example: on a monsoon day, R = 1 if it rains and U = 1 if a student carries an umbrella. The joint table is $P(R{=}0, U{=}0) = 0.56$, $P(R{=}0, U{=}1) = 0.14$, $P(R{=}1, U{=}0) = 0.03$, $P(R{=}1, U{=}1) = 0.27$.

**By hand first:** find $P(R = 1)$, $P(U = 1)$, $P(U = 1 \mid R = 1)$ and $P(R = 1 \mid U = 1)$. Are R and U independent?

In [ ]:
joint = np.array([[0.56, 0.14],      # rows: R = 0, 1;  columns: U = 0, 1
                  [0.03, 0.27]])
pR, pU = joint.sum(axis=1), joint.sum(axis=0)
print(f"P(R=1) = {pR[1]:.2f}, P(U=1) = {pU[1]:.2f}")
print(f"P(U=1 | R=1) = {joint[1, 1] / pR[1]:.4f},  P(R=1 | U=1) = {joint[1, 1] / pU[1]:.4f}  (not the same thing)")
print("P(R) P(U) for every pair:\n", np.outer(pR, pU), "\nindependent:", np.allclose(joint, np.outer(pR, pU)))

outcome = rng.choice(4, size=200_000, p=joint.ravel())        # simulate days from the joint table (0..3 = R*2 + U)
R_sim, U_sim = outcome // 2, outcome % 2
print(f"simulated 200,000 days: P(U=1 | R=1) = {U_sim[R_sim == 1].mean():.4f}, P(R=1 | U=1) = {R_sim[U_sim == 1].mean():.4f}")

### 4. Bayes' rule and the base rate
**Bayes' rule** turns one conditional probability around:
$$P(D \mid +) = \frac{P(+ \mid D)\, P(D)}{P(+ \mid D)\,P(D) + P(+ \mid \text{no } D)\,P(\text{no } D)} .$$
A disease affects 1% of people (the **base rate** or prior). A test finds 99% of sick people ($P(+ \mid D) = 0.99$, the sensitivity) and wrongly flags 1% of healthy people ($P(+ \mid \text{no } D) = 0.01$, the false-positive rate).

**By hand first, with counts:** of 10,000 people, how many are sick, how many of them test positive, and how many healthy people test positive? What fraction of the positives are sick?

In [ ]:
def posterior(prior, sens=0.99, fpr=0.01):
    return sens * prior / (sens * prior + fpr * (1 - prior))

print(f"counts out of 10,000: sick {10_000 * 0.01:.0f}, true positives {100 * 0.99:.0f}, false positives {9_900 * 0.01:.0f}"
      f" -> P(D | +) = 99 / 198 = {99 / 198:.2f}")
print(f"Bayes' rule: P(D | +) = {posterior(0.01):.4f}")

N = 1_000_000
sick = rng.random(N) < 0.01
positive = np.where(sick, rng.random(N) < 0.99, rng.random(N) < 0.01)
print(f"simulated {N:,} people: {positive.sum():,} positives, P(D | +) = {sick[positive].mean():.4f}, P(+ | D) = {positive[sick].mean():.4f}")

for prior in [0.001, 0.01, 0.1, 0.5]:
    print(f"base rate {prior:5.3f}: P(D | +) = {posterior(prior):.4f}")
print(f"after a second, independent positive test (the posterior 0.5 becomes the new prior): P(D | ++) = {posterior(posterior(0.01)):.4f}")

priors = np.logspace(-4, 0, 200)
fig, ax = plt.subplots(figsize=(6, 3.3))
ax.semilogx(priors, posterior(priors), label="P(D | +), test with 99% sensitivity, 1% false positives")
ax.scatter([0.001, 0.01, 0.1, 0.5], [posterior(q) for q in [0.001, 0.01, 0.1, 0.5]], color="black", zorder=3, s=15)
ax.axhline(0.99, color="gray", ls=":", label="P(+ | D) = 0.99")
ax.set(title="The same test means different things at different base rates", xlabel="base rate P(D)", ylabel="probability")
ax.legend(fontsize=7, loc="lower right"); plt.tight_layout(); plt.show()

### 5. The multivariate Gaussian, covariance ellipses and Cholesky sampling
A Gaussian vector $x \sim \mathcal{N}(\mu, \Sigma)$ has mean $\mu$ and **covariance matrix** $\Sigma$, with $\Sigma_{ij} = \operatorname{Cov}(x_i, x_j)$. Its density is constant on the ellipses $(x - \mu)^\top \Sigma^{-1}(x - \mu) = r^2$, whose axes point along the eigenvectors of $\Sigma$ with half-lengths $r\sqrt{\lambda_i}$. In two dimensions a fraction $1 - e^{-r^2/2}$ of the probability lies inside the ellipse of size $r$.

To sample, factor $\Sigma = LL^\top$ with the **Cholesky** factor $L$ (lower triangular), draw $z \sim \mathcal{N}(0, I)$ and set $x = \mu + Lz$. Then $\operatorname{Cov}(x) = L\, I\, L^\top = \Sigma$.

**By hand first:** for $\Sigma$ with rows (4, 2) and (2, 2), check that $L$ with rows (2, 0) and (1, 1) satisfies $LL^\top = \Sigma$. With $\mu = (1, -1)$ and $z = (1, -1)$, what is $x$? What is the correlation $\Sigma_{12}/\sqrt{\Sigma_{11}\Sigma_{22}}$?

In [ ]:
Sigma = np.array([[4.0, 2.0], [2.0, 2.0]])
mu = np.array([1.0, -1.0])
L = np.linalg.cholesky(Sigma)
print("Cholesky factor L =\n", L, "\nL L^T =\n", L @ L.T)
print("x = mu + L z for z = (1, -1):", mu + L @ np.array([1.0, -1.0]), f"; correlation {Sigma[0, 1] / np.sqrt(Sigma[0, 0] * Sigma[1, 1]):.4f}")

Z = rng.standard_normal((5000, 2))
Xs = mu + Z @ L.T                                     # each row is one sample x = mu + L z
print("sample mean", Xs.mean(axis=0), "\nsample covariance (np.cov with rowvar=False)\n", np.cov(Xs, rowvar=False))
print("careful: np.cov(Xs) treats ROWS as variables and returns shape", np.cov(Xs).shape)

lam, Q = np.linalg.eigh(Sigma)
print(f"eigenvalues of Sigma: {lam} (3 - sqrt 5 and 3 + sqrt 5 = {3 - 5 ** 0.5:.4f}, {3 + 5 ** 0.5:.4f})")
d2 = np.einsum("ij,jk,ik->i", Xs - mu, np.linalg.inv(Sigma), Xs - mu)   # squared Mahalanobis distance of each sample
for r in [1, 2, 3]:
    print(f"inside the r = {r} ellipse: {np.mean(d2 <= r * r):.4f} of the samples (theory 1 - exp(-r^2/2) = {1 - np.exp(-r * r / 2):.4f})")

sizes = [50, 500, 5000, 50_000]
cov_err = []
for n in sizes:
    errs = [np.linalg.norm(np.cov(mu + rng.standard_normal((n, 2)) @ L.T, rowvar=False) - Sigma) for _ in range(50)]
    cov_err.append(np.mean(errs))
    print(f"n = {n:6d}: average error of the estimated covariance (Frobenius norm, 50 repeats) {cov_err[-1]:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].scatter(Xs[:, 0], Xs[:, 1], s=2, alpha=0.3, color="gray")
th = np.linspace(0, 2 * np.pi, 200)
for r, c in zip([1, 2, 3], ["tab:blue", "tab:orange", "tab:red"]):
    ell = mu[:, None] + Q @ (r * np.sqrt(lam)[:, None] * np.vstack([np.cos(th), np.sin(th)]))
    ax[0].plot(*ell, lw=1.5, color=c, label=f"r = {r}")
ax[0].set_aspect("equal"); ax[0].legend(fontsize=7)
ax[0].set(title="5,000 samples and the r = 1, 2, 3 ellipses", xlabel="x1", ylabel="x2")
ax[1].loglog(sizes, cov_err, "o-", label="average error")
c_fit = np.exp(np.mean(np.log(cov_err) + 0.5 * np.log(sizes)))          # best constant c in c / sqrt(n)
ax[1].loglog(sizes, c_fit / np.sqrt(sizes), "k:", label="c / sqrt(n) (slope -1/2)")
ax[1].legend(fontsize=8)
ax[1].set(title="Estimating the covariance from samples", xlabel="number of samples n", ylabel="error of the estimate")
plt.tight_layout(); plt.show()

### 6. Correlation is not causation
Hot days raise both cold-drink sales and electricity use (fans and air conditioners). The two are correlated, but neither causes the other: temperature is a **confounder**. We simulate it, then (a) intervene by setting drink sales at random, as an experiment would, and (b) compare only days with almost the same temperature.

In [ ]:
n = 10_000
temp = rng.standard_normal(n)                            # standardized temperature
drinks = temp + rng.standard_normal(n)
power = temp + rng.standard_normal(n)
print(f"correlation of drink sales and electricity use: {np.corrcoef(drinks, power)[0, 1]:.3f}")
drinks_set = rng.standard_normal(n) * np.sqrt(2)          # intervention: sales set at random, not by temperature
print(f"after setting drink sales at random: {np.corrcoef(drinks_set, power)[0, 1]:.3f}")
same_temp = np.abs(temp) < 0.1
print(f"only days with |temperature| < 0.1 ({same_temp.sum()} days): {np.corrcoef(drinks[same_temp], power[same_temp])[0, 1]:.3f}")

### 7. TODO
1. Two positive tests: simulate one million people who take two tests that are independent given the disease status, and estimate $P(D \mid + +)$. Compare with the formula above (0.9900).
2. Sample 10,000 points from a 3-D Gaussian with $\mu = 0$ and $\Sigma$ with rows (4, 2, 0), (2, 3, 1), (0, 1, 2) using the Cholesky factor, and compare `np.cov(samples, rowvar=False)` with $\Sigma$.

In [ ]:
# TODO: sampling a 3-D Gaussian with the Cholesky factor
Sigma3 = np.array([[4.0, 2.0, 0.0], [2.0, 3.0, 1.0], [0.0, 1.0, 2.0]])

def sample_gaussian(mean, cov, n, rng):
    return None      # replace: L = np.linalg.cholesky(cov); return mean + rng.standard_normal((n, len(mean))) @ L.T

S = sample_gaussian(np.zeros(3), Sigma3, 10_000, rng)
print("not done yet" if S is None else np.cov(S, rowvar=False))

### What to notice
PLACEHOLDER